# 02_07 — EMT tiempo real

Diagnóstico inicial de la fuente viva SOAP de ocupación EMT/off-street.

- Fuente viva con cobertura parcial: solo aparcamientos adheridos y con respuesta disponible en el momento de consulta.
- Join principal por `id_emt`; no se fuerza correspondencia por nombre.
- `free_valid` solo conserva valores `free_raw >= 0`.
- La ausencia de dato vivo no se interpreta como ausencia de ocupación.
- Esta capa no se integra todavía en el mapa.

In [1]:
from pathlib import Path
import sys

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontró data_catalog.csv en los padres.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/hugo/TFM_parking_madrid')

In [2]:
from src.data.emt_realtime import build_emt_realtime_from_api

INVENTORY_PATH = ROOT / "data/processed/core/emt/inventario_global_emt.parquet"
RAW_XML_OUTPUT_PATH = ROOT / "data/raw/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_aparcamientos_rotacionales_tiempo_real__latest.xml"
INTERIM_OUTPUT_PATH = ROOT / "data/interim/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_aparcamientos_rotacionales_tiempo_real_latest.parquet"
JOINED_OUTPUT_PATH = ROOT / "data/interim/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_realtime_inventory_join_latest.parquet"

paths = {
    "inventory": INVENTORY_PATH,
    "raw_xml": RAW_XML_OUTPUT_PATH,
    "interim": INTERIM_OUTPUT_PATH,
    "joined": JOINED_OUTPUT_PATH,
}
paths

{'inventory': PosixPath('/Users/hugo/TFM_parking_madrid/data/processed/core/emt/inventario_global_emt.parquet'),
 'raw_xml': PosixPath('/Users/hugo/TFM_parking_madrid/data/raw/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_aparcamientos_rotacionales_tiempo_real__latest.xml'),
 'interim': PosixPath('/Users/hugo/TFM_parking_madrid/data/interim/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_aparcamientos_rotacionales_tiempo_real_latest.parquet'),
 'joined': PosixPath('/Users/hugo/TFM_parking_madrid/data/interim/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_realtime_inventory_join_latest.parquet')}

## Ejecución

La consulta escribe snapshots `latest`. No se acumulan históricos por timestamp en esta fase.

El join con inventario se guarda en `data/interim/emt/emt_aparcamientos_rotacionales_tiempo_real/` porque todavía es una salida diagnóstica viva, no una tabla core final.

In [3]:
result = build_emt_realtime_from_api(
    inventory_path=INVENTORY_PATH,
    raw_xml_output_path=RAW_XML_OUTPUT_PATH,
    interim_output_path=INTERIM_OUTPUT_PATH,
    joined_output_path=JOINED_OUTPUT_PATH,
)

## Diagnóstico

In [4]:
result.metadata

{'endpoint_url': 'https://servayto.madrid.es/MTPAR_WSINFO/InfoParking',
 'language': 'ES',
 'query_timestamp_utc': '2026-07-08T11:03:31.806585+00:00',
 'http_status': 200,
 'response_bytes': 51265,
 'soap_action': 'http://tempuri.org/iInfoParking/GetListParking',
 'n_realtime_rows': 75,
 'n_joined_rows': 85,
 'source': 'emt_realtime_api'}

In [5]:
result.checks

,check_id,status,detail,critical
0,api_http_200,OK,HTTP 200,True
1,api_response_not_empty,OK,51265 bytes,True
2,api_xml_parseable,OK,XML parseable,True
3,realtime_not_empty,OK,75 filas,True
4,realtime_id_not_null,OK,nulos=0,True
5,realtime_id_unique,OK,ids_unicos=75,True
6,realtime_coordinates_parseable,OK,"lat_no_nulas=75, lon_no_nulas=75",False
7,realtime_free_raw_present_some,OK,free_raw_no_nulo=25,False
8,realtime_free_negative_count,OK,free_raw_negativo=0,False
9,inventory_file_exists,OK,/Users/hugo/TFM_parking_madrid/data/processed/...,True


In [6]:
coverage = result.diagnostics["coverage_summary"].iloc[0]

coverage_readable = pd.DataFrame(
    [
        (
            "parkings devueltos por API",
            int(coverage["n_realtime_total"]),
            "Registros presentes en la respuesta SOAP actual.",
        ),
        (
            "parkings API con free_raw",
            int(coverage["n_realtime_with_free_raw"]),
            "Aparcamientos con valor bruto de plazas libres informado.",
        ),
        (
            "parkings API con free_valid",
            int(coverage["n_realtime_with_live_free"]),
            "Aparcamientos con free_raw no negativo y usable como ocupación viva.",
        ),
        (
            "entidades del inventario global",
            int(coverage["n_inventory_total"]),
            "Entidades físicas del inventario integrado EMT/off-street.",
        ),
        (
            "entidades del inventario con id_emt",
            int(coverage["n_inventory_with_id_emt"]),
            "Entidades enlazables por identificador EMT.",
        ),
        (
            "matches inventario-realtime",
            int(coverage["n_inventory_matched_realtime"]),
            "Entidades del inventario con respuesta en la API actual.",
        ),
        (
            "matches con ocupación viva",
            int(coverage["n_inventory_matched_live_free"]),
            "Entidades del inventario con free_valid disponible.",
        ),
        (
            "API sin match inventario",
            int(coverage["n_realtime_only"]),
            "IDs devueltos por la API que no enlazan con inventario_global_emt.",
        ),
        (
            "inventario con id_emt sin realtime",
            int(coverage["n_inventory_only"]),
            "IDs del inventario sin respuesta en la API actual.",
        ),
        (
            "porcentaje de ocupación viva sobre inventario total",
            f"{coverage['pct_inventory_total_matched_live_free']:.1%}",
            "Cobertura viva respecto a todas las entidades del inventario.",
        ),
        (
            "porcentaje de ocupación viva sobre inventario con id_emt",
            f"{coverage['pct_inventory_matched_live_free']:.1%}",
            "Cobertura viva respecto a entidades enlazables por id_emt.",
        ),
        (
            "porcentaje de parkings vivos API que enlazan con inventario",
            f"{coverage['pct_realtime_live_matched_inventory']:.1%}",
            "Parte de los parkings con free_valid que queda integrada por id_emt.",
        ),
    ],
    columns=["metrica", "valor", "lectura"],
)
coverage_readable


,metrica,valor,lectura
0,parkings devueltos por API,75,Registros presentes en la respuesta SOAP actual.
1,parkings API con free_raw,25,Aparcamientos con valor bruto de plazas libres...
2,parkings API con free_valid,25,Aparcamientos con free_raw no negativo y usabl...
3,entidades del inventario global,85,Entidades físicas del inventario integrado EMT...
4,entidades del inventario con id_emt,83,Entidades enlazables por identificador EMT.
5,matches inventario-realtime,66,Entidades del inventario con respuesta en la A...
6,matches con ocupación viva,21,Entidades del inventario con free_valid dispon...
7,API sin match inventario,9,IDs devueltos por la API que no enlazan con in...
8,inventario con id_emt sin realtime,17,IDs del inventario sin respuesta en la API act...
9,porcentaje de ocupación viva sobre inventario ...,24.7%,Cobertura viva respecto a todas las entidades ...


La API es una fuente viva. Las cifras pueden cambiar entre ejecuciones. La diferencia entre parkings con ocupación viva en API y matches vivos contra inventario se explica por cobertura parcial y por aparcamientos presentes en la API que no enlazan por `id_emt` con `inventario_global_emt`.

Cuando esta fuente se integre en el mapa, EMT tiempo real deberá referirse a la hora real de consulta (`query_timestamp`/`moment`), no al intervalo SER redondeado usado por el proxy SER.

In [7]:
result.diagnostics["coverage_status_summary"]


,coverage_status,n
0,realtime_sin_ocupacion_viva,45
1,ocupacion_viva,21
2,inventario_sin_realtime,17
3,sin_id_emt,2


In [8]:
result.diagnostics["matched_live"][[
    "parking_uid",
    "id_emt",
    "nombre",
    "free_raw",
    "free_valid",
    "moment",
    "query_timestamp",
]].head(30)

,parking_uid,id_emt,nombre,free_raw,free_valid,moment,query_timestamp
0,MUN_13452,12,Almagro,54.0,54.0,2026-07-08 13:01:35+02:00,2026-07-08 11:03:31.806585+00:00
4,MUN_5898885,7,Avenida de Portugal,105.0,105.0,2026-07-08 13:01:04+02:00,2026-07-08 11:03:31.806585+00:00
5,MUN_11483771,79,Aviación Española,84.0,84.0,2026-07-08 13:02:49+02:00,2026-07-08 11:03:31.806585+00:00
22,MUN_13459,15,Jacinto Benavente,0.0,0.0,2026-07-08 13:01:18+02:00,2026-07-08 11:03:31.806585+00:00
26,MUN_6143690,25,Marqués de Salamanca,103.0,103.0,2026-07-08 13:00:33+02:00,2026-07-08 11:03:31.806585+00:00
31,MUN_88141,99,Museo de la Ciudad,127.0,127.0,2026-07-08 13:01:40+02:00,2026-07-08 11:03:31.806585+00:00
33,MUN_5517342,22,Orense,46.0,46.0,2026-07-08 13:00:51+02:00,2026-07-08 11:03:31.806585+00:00
34,MUN_52114,71,Pedro Zerolo 'antes denominado Vázquez de Mella',0.0,0.0,2026-07-08 13:01:22+02:00,2026-07-08 11:03:31.806585+00:00
35,MUN_11413620,76,Pitis,176.0,176.0,2026-07-08 13:02:35+02:00,2026-07-08 11:03:31.806585+00:00
38,MUN_13463,53,Plaza de España,267.0,267.0,2026-07-08 13:00:55+02:00,2026-07-08 11:03:31.806585+00:00


In [9]:
result.diagnostics["realtime_only_live"][[
    "id_emt",
    "name",
    "address",
    "free_raw",
    "free_valid",
]].head(30)


,id_emt,name,address,free_raw,free_valid
44,72,Sánchez Bustillo,C/Doctor Mata,46.0,46.0
63,92,El Corte Inglés Preciados,Calle Preciados,149.0,149.0
66,95,Alcala - Sevilla,CALLE DE ALCALÁ,148.0,148.0
71,100,Fuencarral,Cl. Fuencarral,146.0,146.0


In [10]:
result.diagnostics["realtime_only"][[
    "id_emt",
    "name",
    "address",
    "free_raw",
    "free_valid",
]].head(30)


,id_emt,name,address,free_raw,free_valid
7,11,Vázquez de Mella,"Plaza Vázquez de Mella, s/n",NaN,NaN
42,68,Garaje Centro,"C/ Relatores, 11",NaN,NaN
44,72,Sánchez Bustillo,C/Doctor Mata,46.0,46.0
63,92,El Corte Inglés Preciados,Calle Preciados,149.0,149.0
64,93,Escuelas de San Antón,SANTA BRIGIDA,NaN,NaN
66,95,Alcala - Sevilla,CALLE DE ALCALÁ,148.0,148.0
71,100,Fuencarral,Cl. Fuencarral,146.0,146.0
73,102,Pitis,Calle Gloria Fuertes,NaN,NaN
74,103,FuenteMora,Calle Dulce Chacon,NaN,NaN


In [11]:
result.diagnostics["negative_free"][[
    "id_emt",
    "name",
    "free_raw",
    "moment",
]].head(30)


,id_emt,name,free_raw,moment


In [12]:
result.outputs

,output,path,exists,size_mb
0,raw_xml,/Users/hugo/TFM_parking_madrid/data/raw/emt/em...,True,0.049
1,interim_realtime,/Users/hugo/TFM_parking_madrid/data/interim/em...,True,0.016
2,joined_inventory,/Users/hugo/TFM_parking_madrid/data/interim/em...,True,0.041
